<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-Text-Classification/blob/main/huggingface_text_classification_tutorial_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Hugging Face Text Classification Tutorial

* Resources notebook: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial
* Setup steps: https://www.learnhuggingface.com/extras/setup

**Note** A GPU is needed on Google Colab, go to Runtime -> Change runtime -> Hardware accelerator -> GPU.

### Import necessary libraries

In [1]:
import transformers

In [2]:
# Install dependencies (this is mostly for Google Colab)
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using torch version: {torch.__version__}" )
print(f"Using datasets version: {datasets.__version__}")


Using transformers version: 5.16.1
Using torch version: 2.11.0+cu128
Using datasets version: 5.0.1


## 3. Getting a dataset

Building food not food text classification model: need food not food text dataset.

In [3]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

README.md:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.9kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [4]:
# What features are there ?

dataset.column_names

{'train': ['text', 'label']}

In [5]:
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [6]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Import random samples


In [7]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"Text: {text} | Label: {label}")

[INFO] Random samples from dataset:

Text: Comforting lamb curry bowl, featuring tender lamb slow-cooked in a flavorful sauce with cumin and coriander, garnished with toasted cumin seeds. | Label: food
Text: Fusion sushi roll with ingredients like cream cheese or teriyaki sauce. | Label: food
Text: Sushi platter showcasing a variety of colorful rolls and garnishes. | Label: food
Text: Birdhouse hanging from a tree | Label: not_food
Text: A child playing with a golden retriever in the backyard | Label: not_food


In [8]:
# Get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [9]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [10]:
# Turn our dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
159,A close-up of a woman practicing yoga in the l...,not_food
86,"A fruit kabob with a variety of fruits, such a...",food
177,"Green beans in a bowl, sprinkled with almonds ...",food
174,"Artichokes in a bowl, sprinkled with garlic an...",food
141,A slice of pizza with a spicy buffalo chicken ...,food
1,Set of books stacked on a desk,not_food
76,Set of bowls stacked on a shelf,not_food


In [11]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

We want to:

1. Tokenize our text -> turn our text into numbers (this goes for labels as well).
2. Create a train/test split -> want to train our model on the training split and want to evaluate our model on the test split.

In [12]:
# Create a mapping for labels to numeric value

id2label = {0: "not_food", 1: "food"}
label2id = {"not_food" : 0, "food" : 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [13]:
# Create mappings programmatically from dataset
id2label = {idx : label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
id2label

{0: 'not_food', 1: 'food'}

In [14]:
label2id = {label : idx for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id

{'not_food': 0, 'food': 1}

In [15]:
# turn labels into 0 or 1

def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample = {"text" : "This is a sentence about my favprite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favprite food: honey', 'label': 1}

In [16]:
# Map our dataset labels to numbers (the whole thing)
# We do this with dataset.map()  - https://huggingface.co/docs/datasets/process#map
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [17]:
# Shuffle data and look at more 5 random examples
dataset.shuffle()[:5]

{'text': ['Potted plant adding greenery to a windowsill',
  'Working from home at her desk, a woman deals with a cat sitting on the keyboard',
  'Pizza with a white sauce base, topped with spinach and artichokes',
  'Pizza with a dessert twist, featuring a sweet Nutella base and fresh strawberries on top',
  'Set of measuring spoons hung on a rack'],
 'label': [0, 0, 1, 1, 0]}

### Split the dataset into training and test sets

* Train set = model will learn patters on this dataset
* Validation set (optional) = we can tune our model's hyperparameters on this set
* Test set = model will evaluate patters on this dataset

We can split our dataset using `datasets.Dataset.train_test_split`. https://huggingface.co/docs/datasets/v4.8.4/process#split

In [18]:
# Split our dataset into train/test splits

dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [19]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Working from home at her desk, a woman deals with a cat sitting on the keyboard',
 'label': 0}

In [20]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["test"][random_idx_test]
random_sample_test

{'text': 'Artichokes in a bowl, sprinkled with garlic and served with a side of lemon aioli for a tasty, sophisticated dish.',
 'label': 1}

### Tokenizing our text data (turning text into numbers)

The premise of tokenization is to turn words into numbers.

e.g. "I love pizza!" -> [30, 145, 678, 999]

-

The `transformers` library has in-built support for Hugging Face `tokenizers`.
And the class `transformers.AutoTokenizer` helps pair a model to a tokenizer.

-

* To find all models: https://huggingface.co/models
* Model/tokenizer we're going to use: https://huggingface.co/distilbert/distilbert-base-uncased
* Models are aften paired with tokenizers
* Tokenizer = turn text into numbers
* Model = finds patterns in those numbers

In [21]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert-base-uncased",
                                          use_fast=True) # use the fast implementation (on by default, note:this requires Rust installed)
tokenizer


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [22]:
# Test out the tokenizer
tokenizer("I love pizza")

{'input_ids': [101, 1045, 2293, 10733, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

* `input_ids` = our text turned into numbers
* `attention_mask` = whether or not to pay attention to certain tokens (1 = yes pay attention, 0 = no don't pay attention)



In [23]:
# Get the lenght of our tokenizer vocab
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"[INFO] Number of items in our tokenizer vocab: {length_of_tokenizer_vocab}")

# Get the maximum sequence lenght the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"[INFO] Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}")

[INFO] Number of items in our tokenizer vocab: 30522
[INFO] Max tokenizer input sequence length: 512


In [24]:
# Does "daniel" occur in the vocab?
tokenizer.vocab["daniel"]

3817

In [25]:
tokenizer.vocab["x"]

1060

In [26]:
tokenizer("akash")

{'input_ids': [101, 9875, 4095, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [27]:
tokenizer.convert_ids_to_tokens(tokenizer("akash").input_ids)

['[CLS]', 'aka', '##sh', '[SEP]']

In [28]:
# Try to tokenize an emoji
tokenizer.convert_ids_to_tokens(tokenizer("pizza").input_ids)

['[CLS]', 'pizza', '[SEP]']

In [29]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

[('!', 999), ('"', 1000), ('#', 1001), ('##!', 29612), ('##"', 29613)]

In [30]:
import random

random.sample(sorted(tokenizer.vocab.items()), k=5)

[('eventual', 9523),
 ('bobbie', 27731),
 ('wounds', 8710),
 ('shuffling', 24770),
 ('##meral', 28990)]

### Making a preprocessing function to tokenize text

what to make it easy to go from sample -> tokenized_sample


In [31]:
def tokenize_text(examples):
  """
  Tokenize given example text and return the tokenized text.
  """
  return tokenizer(examples["text"],
                   padding = True, # pad short sequences to longest sequence length in batch (e.g., if sample lenght=100, sample will be padded to 512 or longest sample in batch)
                   truncation = True # truncate long sequences to the maximum length the model can handle (e.g., if sample length = 1000, model length = 512, sample will be shortened to 512)
                   )

Extra resource: padding and truncation in the docs: https://huggingface.co/docs/transformers/main/en/pad_truncation

In [32]:
example_sample_2 = {"text" : "I love pizza", "label" : 1}

# Test the function
tokenize_text(example_sample_2)

{'input_ids': [101, 1045, 2293, 10733, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [33]:
long_test = "I love pizza " * 1000
len(long_test)

13000

In [34]:
tokenized_long_text = tokenize_text({"text" : long_test, "label" : 1})
tokenized_long_text

{'input_ids': [101, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293, 10733, 1045, 2293,

In [35]:
len(tokenized_long_text["input_ids"])

512

In [36]:
# Map our tokenize text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True, #Set to tokenize across batches of samples at a time rather than one at time
                                batch_size=1000)
tokenized_dataset

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50
    })
})

**Note:** In machine learning, it is often faster to do things in batches rather than one at a time due to leveraging computer hardware parallelization. See more in the map documentation: https://huggingface.co/docs/datasets/v2.1.0/en/process#map

In [37]:
tokenizer.all_special_tokens

['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']

In [38]:
tokenizer.all_special_ids

[100, 102, 0, 101, 103]

In [39]:
# Get two samples from the tokenized datasets
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]

for key in train_tokenized_sample.keys():
  print(f"[INFO] Key:")
  print(f"Train sample: {train_tokenized_sample[key]}")
  print(f"Test sample: {test_tokenized_sample[key]}")
  print("\n")

[INFO] Key:
Train sample: Set of headphones placed on a desk
Test sample: A slice of pepperoni pizza with a layer of melted cheese


[INFO] Key:
Train sample: 0
Test sample: 1


[INFO] Key:
Train sample: [101, 2275, 1997, 2132, 19093, 2872, 2006, 1037, 4624, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [101, 1037, 14704, 1997, 11565, 10698, 10733, 2007, 1037, 6741, 1997, 12501, 8808, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


[INFO] Key:
Train sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


[INFO] Key:
Train sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

### Tokenization takeaways

1. Tokenizers = turn data into numbers (e.g. text -> map to number)
2. Many models are out there and have different tokenizers, Hugging Face's Auto (e.g., AutoTokenizer, AutoProcessor, Automodel etc help to match tokenizers to models)
3. Tokenization can happen in parallel using map and batched functions

## Settinng up an evaluation metric

What we want to do: use the evaluation metric to get a numerical idea of how our model is performing.

Some common evaluation metrics for classification:

- Accuracy (how many examples out of 100, did you get correct ?)
- Precision
- Recall
- F1 Score

Evaluation metric is important because some projects may have en evaluation threshold you need to fulfill.

E.g., when I worked on insurance claim classification = require 98% + test accuracy to br commercially viable.

Some places for evaluation metrics:

* Scikit-learn documentation: https://scikit-learn.org/stable/modules/model_evaluation.html
* Hugging Face evaluate: https://huggingface.co/docs/evaluate/types_of_evaluations

In [40]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.ndarray, np.ndarray]):
  """
  Compute the accuracy of  model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

In [41]:
# Example predictions and accuracy score
example_preds_all_correct = np.array([0,0,0,0,0,0,0,0,0,0])
example_preds_one_incorrect = np.array([0,0,0,1,0,0,0,0,0,0])
example_labels = np.array([0,0,0,0,0,0,0,0,0,0])

# test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_preds_all_correct, example_labels))}")
print(f"Accuracy when all predictions are incorrect: {compute_accuracy((example_preds_one_incorrect, example_labels))}")

Accuracy when all predictions are correct: {'accuracy': 1.0}
Accuracy when all predictions are incorrect: {'accuracy': 0.9}


## Setting our model to training

- We're going to be using transfer learning.
- Transfer learning is a powerfulk technique, unique to deep learning models that enables us to use the patterns one model has learned on another problem for our own problem.
- See more on transfer learning here: https://en.wikipedia.org/wiki/Transfer_learning

Workflow for training:

1. Create and preprocess data
2. Define the model we'd like to use for our problem: https://huggingface.co/models or see the "task guides" in the HF Transformers docs: https://huggingface.co/tasks/text-generation
3. Define training arguments for training our model `transformers.TrainingArguments`
4. Pass `TrainingArguments` to an instance of `transformers.Trainer`
5. Train the model by calling `Trainer.train()`
6. Save the model (to our local machine or to Hugging Face Hub)
7. Evaluate the trained model by making and inspecting predictions on the test data (and our own custom data of course)
8. Turn teh model into a shareable demo

In [42]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


Our model is comprised of the following parts:

1. `embeddings` - embeddings are a form of learned representation of tokens. So if tokens are a direct mapping from token to number, embeddings are a learned vector representation.
2. `transformer` - our model architecture backbone, this has discovered patterns/relationships in the embeddings.
3. `classifier` - we need to custimize this layer to suit our problem.

**Note:** If you get input errors from passing a sample to a model, make sure the sample you pass to your model is formatting in the same way you model was trained on. For example, if your model used a specific tokenizer, make sure to tokenize your text before passing it to the model.

### Count parameters in our model

Weights/parameters = small numeric opportunities for a model to learn patterns in data.

In [44]:
model.parameters()

<generator object Module.parameters at 0x7e3f181335a0>

In [45]:
def count_params(model):
  """
  Count the parameters in a PyTorch model
  """
  trainable_parameters = sum(param.numel() for param in model.parameters() if param.requires_grad)
  total_parameters = sum(param.numel() for param in model.parameters())

  return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

count_params(model)

{'trainable_parameters': 66955010, 'total_parameters': 66955010}

Looks like our model has around 67M parameters and **all** of them are trainable.

Note:
* Generally, the more parameters a model has, the more capacity is has to learn.
* For comparison models such as Llama 3, it has 8 billion parameters.
* If you want the best possible performance, generally more parameters is better.
  * However, with more parameters requires more compute + time.
  * You'll be surprised how well a smaller model can perform with specific data.

### Create a directory for saving models

In [46]:
# Create a model output directory

from pathlib import Path

# Create models dir
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

#Create model save name
model_save_name ="learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)
model_save_dir


PosixPath('models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased')

### Setting up training arguments (hyperparameters) with TrainingArguments

3. Define training arguments for training our model `transformers.TrainingArguments`
  * These are also known as "hyperparameters" = settings on your model that you can adjust
  * Parameters = weights/patterns in the model that get updated automatically

In [47]:
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints: {model_save_dir}")

BATCH_SIZE = 32

# Creating training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs = 10,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit=3,
    use_cpu = False,
    seed = 42,
    load_best_model_at_end=True,
    logging_strategy = "epoch",
    report_to = "none",
    # hub_token = "Your token her"
    # push_to_hub = True if your want your model to save directly to the Hugging Face after training
    hub_private_repo = False # when uploading to Hugging Face Hub, do you want your repo to be private or public? (default: public)
)

[INFO] Saving model checkpoints: models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased


In [48]:
training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

* Docs fro transformers.Trainer: https://huggingface.co/docs/transformers/peft

In [49]:
compute_accuracy

<function __main__.compute_accuracy(predictions_and_labels: Tuple[numpy.ndarray, numpy.ndarray])>

In [50]:
tokenized_dataset["train"]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})

In [51]:
from transformers import Trainer

# Setup Trainer instances
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_accuracy
)

trainer

In [56]:
results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.019337,0.012037,1.000000
2,0.012680,0.008559,1.000000
3,0.009384,0.006691,1.000000
4,0.007553,0.005603,1.000000
5,0.006472,0.004911,1.000000
6,0.005822,0.004464,1.000000
7,0.005354,0.004169,1.000000
8,0.005226,0.003981,1.000000
9,0.004983,0.003875,1.000000
10,0.004770,0.003838,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [57]:
results.metrics

{'train_runtime': 188.6074,
 'train_samples_per_second': 10.604,
 'train_steps_per_second': 0.371,
 'total_flos': 18110777160000.0,
 'train_loss': 0.008158049945320402,
 'epoch': 10.0}

In [58]:
# Inspect training metrics

for key, value in results.metrics.items():
  print(f"{key} : {value}")

train_runtime : 188.6074
train_samples_per_second : 10.604
train_steps_per_second : 0.371
total_flos : 18110777160000.0
train_loss : 0.008158049945320402
epoch : 10.0


### save the model for later us

> **Note:** if you are saving a model to Google Colab, note that it will desappear from your Colab instances when it disconnets.

In [59]:
# Save model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

[INFO] Saving model to models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]